In [1]:
!pip install -q transformers datasets accelerate

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

In [4]:
data = pd.read_csv('/content/drive/MyDrive/Mimicaught/phishing_email.csv')

df = pd.DataFrame(data)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82486 entries, 0 to 82485
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   text_combined  82486 non-null  object
 1   label          82486 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.3+ MB


In [6]:
print(df.shape)
print(df.columns.tolist())
print(df.head())
print(df.iloc[:, -1].value_counts())      # label balance (adjust if label isn't last)
print("nulls:\n", df.isnull().sum())
print("duplicates:", df.duplicated().sum())

(82486, 2)
['text_combined', 'label']
                                       text_combined  label
0  hpl nom may 25 2001 see attached file hplno 52...      0
1  nom actual vols 24 th forwarded sabrae zajac h...      0
2  enron actuals march 30 april 1 201 estimated a...      0
3  hpl nom may 30 2001 see attached file hplno 53...      0
4  hpl nom june 1 2001 see attached file hplno 60...      0
label
1    42891
0    39595
Name: count, dtype: int64
nulls:
 text_combined    0
label            0
dtype: int64
duplicates: 408


In [7]:
FINGERPRINTS = ["josemonkeyorg", "monkeyorg"]

def prep(text):
    t = str(text).lower()
    t = re.sub(r"[^a-z0-9\s]", "", t)     # keep letters, digits and whitespace
    t = re.sub(r"\d+", " ", t)             # digits -> space
    for f in FINGERPRINTS:
        t = t.replace(f, " ")
    return re.sub(r"\s+", " ", t).strip()

def make_baseline(C=1.0):
    return make_pipeline(
        TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=3, sublinear_tf=True),
        LogisticRegression(max_iter=1000, C=C, class_weight="balanced"),
    )

def show_top_words(model, n=25):
    tfidf = model.named_steps['tfidfvectorizer']
    clf = model.named_steps['logisticregression']
    names = np.array(tfidf.get_feature_names_out())
    order = np.argsort(clf.coef_[0])
    print("legit-leaning:   ", names[order[:n]])
    print("phishing-leaning:", names[order[-n:]])

In [8]:
df["clean"] = df["text_combined"].apply(prep)

X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

model = make_baseline().fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test), target_names=["Legit (0)", "Phishing (1)"]))
show_top_words(model)

              precision    recall  f1-score   support

   Legit (0)       0.99      0.98      0.99      7919
Phishing (1)       0.99      0.99      0.99      8579

    accuracy                           0.99     16498
   macro avg       0.99      0.99      0.99     16498
weighted avg       0.99      0.99      0.99     16498

legit-leaning:    ['wrote' 'enron' 'thanks' 'opensuse' 'university' 'vince' 'im' 'pm'
 'louise' 'perl' 'attached' 'language' 'would' 'python' 'aug opensuse'
 'list' 'url' 'edu' 'questions' 'date' 'linguistics' 'february'
 'following' 'anyone' 'ierant']
phishing-leaning: ['pills' 'utf' 'health' 'mobile email' 'dear' 'phone mobile' 'quality'
 'company' 'viagra' 'meds' 'investment' 'bank' 'wed aug' 'remove' 'men'
 'replica' 'watches' 'httpwwwcnncom' 'click' 'aug' 'money' 'love' 'life'
 'account' 'http']


In [9]:
z = pd.read_csv('/content/drive/MyDrive/Mimicaught/Phishing_validation_emails.csv')
print(z.columns.tolist()); print(z.head(3))

['Email Text', 'Email Type']
                                          Email Text      Email Type
0  Dear Jordan, your subscription has been succes...      Safe Email
1  Dear Casey, thank you for your purchase. Your ...      Safe Email
2  Congratulations! You've won a $3000 gift card....  Phishing Email


In [10]:
base = "/content/drive/MyDrive/Mimicaught/"
files = {"CEAS": "CEAS_08.csv", "Enron": "Enron.csv", "Ling": "Ling.csv",
         "Nazario": "Nazario.csv", "Nigerian": "Nigerian_Fraud.csv", "SpamAssassin": "SpamAssassin.csv"}

frames = []
for src, fn in files.items():
    d = pd.read_csv(base + fn)
    print(src, d.columns.tolist(), d["label"].value_counts().to_dict())
    subject = d["subject"].fillna("").astype(str) if "subject" in d.columns else pd.Series("", index=d.index)
    body    = d["body"].fillna("").astype(str)    if "body" in d.columns    else pd.Series("", index=d.index)
    frames.append(pd.DataFrame({"text": (subject + " " + body).apply(prep),
                                "label": d["label"].astype(int), "source": src}))

big = pd.concat(frames, ignore_index=True).drop_duplicates(subset="text")
big = big[big["text"].str.split().str.len() >= 5]
print(big.groupby("source")["label"].agg(["count", "mean"]))

CEAS ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls'] {1: 21842, 0: 17312}
Enron ['subject', 'body', 'label'] {0: 15791, 1: 13976}
Ling ['subject', 'body', 'label'] {0: 2401, 1: 458}
Nazario ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label'] {1: 1565}
Nigerian ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label'] {1: 3332}
SpamAssassin ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls'] {0: 4091, 1: 1718}
              count      mean
source                       
CEAS          35726  0.520517
Enron         28247  0.491734
Ling           2853  0.160182
Nazario        1528  1.000000
Nigerian       3254  1.000000
SpamAssassin   5745  0.290688


In [11]:
rows = []
for held_out in big["source"].unique():
    train, test = big[big.source != held_out], big[big.source == held_out]
    m = make_baseline().fit(train["text"], train["label"])
    pred = m.predict(test["text"])
    y = test["label"].values
    rows.append({
        "held_out": held_out, "n_test": len(test),
        "phishing_caught": (pred[y == 1] == 1).mean() if (y == 1).any() else np.nan,
        "legit_kept_safe": (pred[y == 0] == 0).mean() if (y == 0).any() else np.nan,
    })
print(pd.DataFrame(rows).round(3))

       held_out  n_test  phishing_caught  legit_kept_safe
0          CEAS   35726            0.965            0.902
1         Enron   28247            0.884            0.717
2          Ling    2853            0.928            0.942
3       Nazario    1528            0.657              NaN
4      Nigerian    3254            0.989              NaN
5  SpamAssassin    5745            0.953            0.919


In [12]:
# 1. does prep work at all?
print(repr(prep("Click HERE to verify 2025 your account now please")))

# 2. look at one real file step by step
d = pd.read_csv(base + "Enron.csv")
raw = d["subject"].fillna("").astype(str) + " " + d["body"].fillna("").astype(str)
cleaned = raw.apply(prep)
print(raw.iloc[0][:150])
print(cleaned.iloc[0][:150])
print("empty after prep:", (cleaned.str.len() == 0).sum(), "of", len(cleaned))
print("unique cleaned texts:", cleaned.nunique())

'click here to verify your account now please'
hpl nom for may 25 , 2001 ( see attached file : hplno 525 . xls )
- hplno 525 . xls
hpl nom for may see attached file hplno xls hplno xls
empty after prep: 2 of 29767
unique cleaned texts: 28306


In [13]:
def source_weights(train):
    counts = train["source"].map(train["source"].value_counts())
    w = 1.0 / np.sqrt(counts)            # softens big sources without ignoring size
    return (w / w.mean()).values

def loso(held_list, weighted):
    rows = []
    for held in held_list:
        train, test = big[big.source != held], big[big.source == held]
        m = make_baseline()
        kw = {"logisticregression__sample_weight": source_weights(train)} if weighted else {}
        m.fit(train["text"], train["label"], **kw)
        pred, y = m.predict(test["text"]), test["label"].values
        rows.append({"held_out": held, "weighted": weighted,
                     "phishing_caught": (pred[y == 1] == 1).mean() if (y == 1).any() else np.nan,
                     "legit_kept_safe": (pred[y == 0] == 0).mean() if (y == 0).any() else np.nan})
    return pd.DataFrame(rows)

print(pd.concat([loso(["Nazario", "Enron"], False), loso(["Nazario", "Enron"], True)]).round(3))

  held_out  weighted  phishing_caught  legit_kept_safe
0  Nazario     False            0.670              NaN
1    Enron     False            0.904            0.784
0  Nazario      True            0.720              NaN
1    Enron      True            0.925            0.739


In [16]:
import joblib

final = make_baseline().fit(
    big["text"], big["label"],
    logisticregression__sample_weight=source_weights(big)
)
joblib.dump(final, "/content/drive/MyDrive/Mimicaught/models_saved/email_baseline.pkl")

['/content/drive/MyDrive/mimicaught/models_saved/email_baseline.pkl']